# Environment

In [ ]:
import gc
import os
import sys
import logging
import warnings
from pathlib import Path
import matplotlib as mpl
import pandas as pd
import session_info

SRCDIR = Path('../../..')
HOMEDIR = SRCDIR / '..'
PLOTDIR = HOMEDIR / "plots" / "kennedi_xenium"
DATADIR = HOMEDIR / "data" / "processed" / "spatial" / "Xenium" / "kennedi_flu"
if not str(SRCDIR) in sys.path:
    sys.path.insert(0, str(SRCDIR))

logging.basicConfig(level="WARNING")
warnings.simplefilter("ignore", FutureWarning)
warnings.simplefilter("ignore", UserWarning)
warnings.simplefilter("ignore", RuntimeWarning)
warnings.simplefilter("ignore", DeprecationWarning)

os.environ['R_HOME'] = "/opt/R/4.4.1"
from single_cell.preprocess import *
from single_cell.plot import *
from single_cell.analysis import *
from spatial_seq.plot import *
from utils import *

import anndata as ad
import spatialdata as sd
import spatialdata_plot as sdp

%matplotlib inline
# R_preload()
mpl.rcdefaults()
gc.collect()

session_info.show()

In [4]:
CLUSTER_KEY = "global_leiden"
DE_KEY = "global_DEG"
resolutions = np.arange(1, 16) / 10

# Load data object

In [ ]:
sdatas = {
    fname.split("__")[1]:
    sd.read_zarr(DATADIR / fname)
    for fname in tqdm(os.listdir(DATADIR), desc="Reading spatialdata objects")
}

REFDIR = Path("../../../../../../../kpyper") / "Xenium_Flu"
data_folders = [folder for folder in REFDIR.rglob("*Molofski_Pyper*") if folder.is_dir()]

annotations = {}
for folder in data_folders:
    for out_folder in [subfolder for subfolder in folder.iterdir() if subfolder.is_dir()]:
        slide = out_folder.name.split('__')[1]
        annotations[slide] = {}
        for file in os.listdir(out_folder / "regions"):
            sample = file.split(".csv")[0]
            df = pd.read_csv(out_folder / "regions" / file, skiprows=2)
            annotations[slide][sample] = df.copy()

In [ ]:
for slide in annotations.keys():
    # switch to cell segmentation
    sdatas[slide]['table'].obs['region'] = "cell_boundaries"
    sdatas[slide]['table'].obs['region'] = sdatas[slide]['table'].obs['region'].astype(str).astype("category")
    sdatas[slide]['table'].uns['spatialdata_attrs']['region'] = "cell_boundaries"

    # annotate by Sample by cell_id
    sdatas[slide]['table'].obs["Slide"] = slide
    sdatas[slide]['table'].obs["Sample"] = None
    for sample in annotations[slide]:
        idx = sdatas[slide]['table'].obs['cell_id'].isin(annotations[slide][sample]['Cell ID'])
        sdatas[slide]['table'].obs.loc[idx,"Sample"] = sample

    # find cells with no Sample annotation
    to_remove = sdatas[slide]['table'].obs["Sample"].isna()
    print(f"Slide {slide}: {sum(to_remove)} cells removed")

    # remove said cells
    sdatas[slide]['table'] = sdatas[slide]['table'][~to_remove]

In [ ]:
sdata = sd.concatenate(sdatas, concatenate_tables=True)

In [ ]:
sample = "0036329"
sdata.pl.render_images(
    f"morphology_focus-{sample}",
    cmap="viridis",
    # channel=["ATP1A1/CD45/E-Cadherin"],
    norm=mpl.colors.LogNorm(),
    ).pl.render_shapes(
        f"cell_circles-{sample}",
        fill_alpha=0.5,
        na_color="blue",
        outline_width=0.3,
        outline_color="white",
).pl.show(coordinate_systems=f"global-{sample}", title="IF image")

### Sample filter on lung tissue

In [ ]:
sdata_crop = sd.bounding_box_query(
    sdata,
    min_coordinate=[7000, 0],
    max_coordinate=[27000, 18000],
    axes=("x", "y"),
    target_coordinate_system="global",
)

sdata_crop.pl.render_images(
    "morphology_focus",
    cmap="viridis",
    channel=["ATP1A1/CD45/E-Cadherin"],
    norm=mpl.colors.LogNorm(),
).pl.render_shapes(
    "cell_circles",
    fill_alpha=0.8,
    na_color="red",
    outline_width=0.01,
    outline_color="black",
).pl.show(
    coordinate_systems="global", title="Cell segmentation"
)

sdata_crop

In [ ]:
# filter based on circles
adipose_cell_ids = pd.read_csv(
    DATADIR / "xenium-038439-adipose_tissue.csv", skiprows=966
)["Cell ID"]
sdata_crop.tables["table"] = sdata_crop["table"][
    sdata_crop["table"].obs["cell_id"].isin(adipose_cell_ids)
]

# filter minimum 10 UMIs each
mask = sdata_crop.tables["table"].obs["transcript_counts"] >= 10
sdata_crop.tables["table"] = sdata_crop.tables["table"][mask]

sdata_crop.shapes, sdata_crop.tables["table"] = sd.match_element_to_table(
    sdata_crop, list(sdata_crop.shapes.keys()), "table"
)

sdata_crop.pl.render_shapes(
    "cell_circles",
    color="transcript_counts",
    # fill_alpha=0.8,
    # na_color="red",
    # outline_width=0.01,
    # outline_color="black",
).pl.show(coordinate_systems="global", title="Cell segmentation")

# Process transcriptomic data

### Single cell analysis

In [ ]:
# read in sequencing
sdatas = {
    fname.split("__")[1]:
    sd.read_zarr(DATADIR / fname)
    for fname in tqdm(os.listdir(DATADIR), desc="Reading spatialdata objects")
}

# read in annotations
REFDIR = Path("../../../../../../../kpyper") / "Xenium_Flu" / "data"
data_folders = [folder for folder in REFDIR.rglob("*Molofski_Pyper*") if folder.is_dir()]
annotations = {}
for folder in data_folders:
    for out_folder in [subfolder for subfolder in folder.iterdir() if subfolder.is_dir()]:
        slide = out_folder.name.split('__')[1]
        annotations[slide] = {}
        for file in os.listdir(out_folder / "regions"):
            sample = file.split(".csv")[0]
            df = pd.read_csv(out_folder / "regions" / file, skiprows=2)
            annotations[slide][sample] = df.copy()

# filter sequencing on annotations
for slide in annotations.keys():
    # switch to cell segmentation
    sdatas[slide]['table'].obs['region'] = "cell_boundaries"
    sdatas[slide]['table'].obs['region'] = sdatas[slide]['table'].obs['region'].astype(str).astype("category")
    sdatas[slide]['table'].uns['spatialdata_attrs']['region'] = "cell_boundaries"

    # annotate by Sample by cell_id
    sdatas[slide]['table'].obs["Slide"] = slide
    sdatas[slide]['table'].obs["Sample"] = None
    for sample in annotations[slide]:
        idx = sdatas[slide]['table'].obs['cell_id'].isin(annotations[slide][sample]['Cell ID'])
        sdatas[slide]['table'].obs.loc[idx,"Sample"] = sample

    # find cells with no Sample annotation
    to_remove = sdatas[slide]['table'].obs["Sample"].isna()
    print(f"Slide {slide}: {sum(to_remove)} cells removed")

    # remove said cells
    sdatas[slide]['table'] = sdatas[slide]['table'][~to_remove]

# combine samples
sdata = sd.concatenate(sdatas, concatenate_tables=True)

In [ ]:
# standard preprocessing
adata = sdata.tables["table"]
adata.layers["counts"] = adata.X.copy()
adata = Filter_QC(adata, 10, 0, 0)
Normalize(adata)
PCA(adata, gene_mask=None, key="PCA", comp=20)

Integrate(adata, "Slide", gene_mask=None, pca_key="PCA")
Visualize(adata, localmap=False)
Cluster(adata, neighbor_key="neighbors", cluster_key=CLUSTER_KEY, resolutions=resolutions)

# Load after integration

In [ ]:
sdata = sd.read_zarr(DATADIR / "integrated.zarr")
sdata

In [ ]:
clear_adata(sdata['table'], ["dendrogram", "colors", "X_pca"])
sd.sanitize_table(sdata['table'])
sdata.shapes, sdata["table"] = sd.match_element_to_table(
    sdata, list(sdata.shapes.keys()), "table"
)

In [ ]:
sdata['table'].copy().write(DATADIR / "integrated.h5ad")

In [11]:
adata = sc.read_h5ad(DATADIR / "integrated.h5ad")

# Integrate back into spatialdata object

In [ ]:
clear_adata(adata, ["dendrogram", "colors", "X_pca"])
sd.sanitize_table(adata)
sdata_crop.shapes, sdata_crop.tables["table"] = sd.match_element_to_table(
    sdata_crop, list(sdata_crop.shapes.keys()), "table"
)

sdata_crop.pl.render_images(
    "morphology_focus",
    cmap="gray_r",
    # channel=['ATP1A1/CD45/E-Cadherin'],
    norm=mpl.colors.LogNorm(),
).pl.render_shapes(
    "cell_circles",
    color="leiden_0.5",
    # fill_alpha=0.3,
    # na_color="red",
    outline_width=0.01,
    outline_color="black",
).pl.show(
    coordinate_systems="global", title="Clusters"
)

In [ ]:
# switch table annotation
adata.obs["region"] = "cell_boundaries"
adata.obs["region"] = adata.obs["region"].astype("category")
adata.uns["spatialdata_attrs"]["region"] = "cell_boundaries"

In [ ]:
t1_adata = sd.bounding_box_query(
    sdata_crop,
    min_coordinate=[7500, 0],
    max_coordinate=[13000, 6000],
    axes=("x", "y"),
    target_coordinate_system="global",
)

axs = plt.subplots(1, 2, figsize=(20, 10))[1]

for i, col in enumerate(["leiden_0.5", "leiden_0.8"]):
    t1_adata.pl.render_images(
        "morphology_focus",
        cmap="gray_r",
        # channel=['ATP1A1/CD45/E-Cadherin'],
        norm=mpl.colors.LogNorm(),
    ).pl.render_shapes(
        "cell_boundaries",
        color=col,
        fill_alpha=1,
        # na_color="red",
        outline_width=0.01,
        outline_color="black",
    ).pl.show(
        ax=axs[i], coordinate_systems="global", title="Clusters"
    )

# SAVE / LOAD

In [ ]:
sdata_crop.write(DATADIR / "0038439_adipose_CLEANED.zarr", overwrite=True)

In [ ]:
sdata_crop = sd.read_zarr(DATADIR / "0038439_adipose_CLEANED.zarr")
sdata_crop

# Sandbox

In [ ]:
t1_adata = sd.bounding_box_query(
    sdata_crop,
    min_coordinate=[7500, 0],
    max_coordinate=[13000, 6000],
    axes=("x", "y"),
    target_coordinate_system="global",
)

axs = plt.subplots(1, 2, figsize=(20, 10))[1]

for i, col in enumerate(["leiden_0.5", "leiden_0.8"]):
    t1_adata.pl.render_images(
        "morphology_focus",
        cmap="gray_r",
        # channel=['ATP1A1/CD45/E-Cadherin'],
        norm=mpl.colors.LogNorm(),
    ).pl.render_shapes(
        "cell_boundaries",
        color=col,
        fill_alpha=1,
        # na_color="red",
        outline_width=0.01,
        outline_color="black",
    ).pl.show(
        ax=axs[i], coordinate_systems="global", title="Clusters"
    )

In [ ]:
axs = plt.subplots(1, 2, figsize=(20, 10))[1]

for i, layer in enumerate(["normalized", "RECODE_log"]):
    t1_adata.pl.render_images(
        "morphology_focus",
        cmap = "gray_r",
        # channel = ['ATP1A1/CD45/E-Cadherin'],
        norm=  mpl.colors.LogNorm(),
    ).pl.render_shapes(
        "cell_boundaries",
        color = "Pdgfra",
        table_layer = layer,
        fill_alpha = 1,
        # na_color = "red",
        outline_width = 0.01,
        outline_color = "black",
    ).pl.show(
        ax = axs[i], coordinate_systems = "global", title = "Clusters"
    )